# Tessera JAX backend — Tier 3: batched-population vmap eval

**Status (2026-05-24):** Tier 3 has landed.

**The problem Tier 3 solves:** even with Tier 2's jit-compiled trees, a GP generation has to loop over K trees in Python:

```
for tree in population:                # K=200 in Python
    y = evaluate_jit(tree, env)        # one kernel launch each
```

Each kernel launch is ~10-50 μs of dispatch overhead. For K=200, that's 2-10 ms per generation of pure overhead, BEFORE compute. On GPU where individual kernels are fast, this overhead dominates.

**Tier 3's fix:** cluster trees by topology, vmap each cluster into a single jit kernel. K=200 candidates with G=10 topology groups → 10 kernel launches instead of 200. On GPU, batched vmap fuses the work into K parallel SIMD lanes inside one kernel.

**Honest caveats:**
- **On CPU JAX, vmap is overhead, not optimization.** No SIMD lanes to fill; the batch dimension is pure cost. Expect batched-vs-per-tree-jit ratio < 1 on CPU JAX. The win is on GPU.
- **The speedup depends on topology clustering**: 200 trees in 5 topologies → great; 200 trees in 200 different topologies → no batching wins.
- **Trees containing `FunctionalOp` can't be jit-batched** — they fall back to the eager Tier 1 path.

## Setup

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch runtime to GPU'

In [ ]:
!pip install -q git+https://github.com/davechendatascience/tessera.git
!pip install -q --upgrade "jax[cuda12]"

In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import time
import pandas as pd

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Build a representative population

Mimicking late-GP behavior: K=200 trees clustering into ~10 topologies (different constants per tree). This is the realistic case where Tier 3 wins.

In [ ]:
from tessera.expression import (
    Var, Const, BinOp, UnOp, evaluate, evaluate_jit, evaluate_population,
    clear_jit_cache, clear_topo_cache, topo_cache_size, topology_key,
)

# 10 topology templates spanning typical SR shapes
def gen_population(K=200, seed=0):
    rng = np.random.default_rng(seed)
    pop = []
    for _ in range(K):
        topo_choice = rng.integers(0, 10)
        # Random constants per tree
        c1, c2, c3 = [float(rng.standard_normal()) for _ in range(3)]
        if topo_choice == 0:
            t = BinOp('add', BinOp('mul', Var('x'), Const(c1)), Const(c2))
        elif topo_choice == 1:
            t = UnOp('tanh', BinOp('mul', Var('x'), Const(c1)))
        elif topo_choice == 2:
            t = BinOp('sub', UnOp('tanh', Var('x')), Const(c1))
        elif topo_choice == 3:
            t = BinOp('mul', UnOp('sqrt', Var('x')), Const(c1))
        elif topo_choice == 4:
            t = BinOp('add', UnOp('exp', BinOp('mul', Var('x'), Const(c1))), Const(c2))
        elif topo_choice == 5:
            t = BinOp('div', Var('x'), BinOp('add', Var('x'), Const(c1)))
        elif topo_choice == 6:
            t = BinOp('pow', Var('x'), Const(c1))
        elif topo_choice == 7:
            t = UnOp('log', BinOp('add', BinOp('mul', Var('x'), Var('x')), Const(c1)))
        elif topo_choice == 8:
            t = BinOp('mul', BinOp('add', Var('x'), Const(c1)),
                              BinOp('sub', Var('x'), Const(c2)))
        else:
            t = BinOp('add', BinOp('mul', Var('x'), Const(c1)),
                              BinOp('mul', UnOp('tanh', Var('x')), Const(c2)))
        pop.append(t)
    return pop

population = gen_population(K=200)
print(f'Population size: {len(population)}')

from collections import Counter
topo_counts = Counter(topology_key(t) for t in population)
print(f'Distinct topologies: {len(topo_counts)}')
print(f'Cluster sizes: {sorted(topo_counts.values(), reverse=True)}')

## Time three paths: numpy, per-tree-jit, batched-vmap

For each of three N sizes, compute:
1. numpy: loop, evaluate each tree on numpy env
2. per-tree jit: loop, evaluate_jit each tree on JAX env
3. batched: evaluate_population on JAX env

In [ ]:
results = []
for N in [10_000, 60_000, 600_000]:
    rng = np.random.default_rng(0)
    x_np = rng.standard_normal(N).astype(np.float32)
    env_np = {'x': x_np}
    env_jax = {'x': jnp.asarray(x_np)}

    clear_jit_cache(); clear_topo_cache()

    # 1) numpy loop
    _ = [evaluate(t, env_np) for t in population]   # warmup
    t0 = time.time()
    for _ in range(5):
        outs = [evaluate(t, env_np) for t in population]
    t_np = (time.time() - t0) / 5

    # 2) per-tree jit loop (Tier 2)
    _ = [evaluate_jit(t, env_jax).block_until_ready() for t in population]
    t0 = time.time()
    for _ in range(5):
        outs = [evaluate_jit(t, env_jax).block_until_ready() for t in population]
    t_jit = (time.time() - t0) / 5

    # 3) batched (Tier 3)
    _ = [r.block_until_ready() for r in evaluate_population(population, env_jax)]
    t0 = time.time()
    for _ in range(5):
        outs = evaluate_population(population, env_jax)
        for r in outs:
            r.block_until_ready()
    t_batched = (time.time() - t0) / 5

    results.append(dict(
        N=N,
        numpy_ms=t_np * 1000,
        per_tree_jit_ms=t_jit * 1000,
        batched_ms=t_batched * 1000,
        jit_vs_numpy=t_np / t_jit,
        batched_vs_numpy=t_np / t_batched,
        batched_vs_jit=t_jit / t_batched,
    ))

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nTopology cache size: {topo_cache_size()} (expect ~10)")

**Expected reading on GPU:**
- `jit_vs_numpy`: confirms Tier 2's speedup (5-50× from previous notebook)
- `batched_vs_jit`: this is the Tier 3 incremental win. On GPU, expect 2-10× over per-tree jit at K=200 / G=10 (cluster sizes ~20 each)
- `batched_vs_numpy`: the bottom-line speedup. Multiply Tier 2 × Tier 3.

**Expected on CPU JAX:** `batched_vs_jit < 1` because vmap is pure overhead without GPU SIMD lanes. The notebook's purpose is the GPU measurement.

## Correctness: batched outputs match per-tree outputs

In [ ]:
x_np = np.random.default_rng(0).standard_normal(1000).astype(np.float32)
env_jax = {'x': jnp.asarray(x_np)}

outs_batched = evaluate_population(population, env_jax)
outs_per_tree = [evaluate_jit(t, env_jax) for t in population]

max_diff = max(
    float(jnp.max(jnp.abs(b - p))) for b, p in zip(outs_batched, outs_per_tree)
)
print(f'max |batched - per_tree| over all {len(population)} trees: {max_diff:.2e}')
assert max_diff < 1e-3
print('outputs match within float32 precision ✓')

## What's the path to MNIST 95%?

After Tier 1+2+3, here's what a single MNIST GP generation costs (estimated, K=200, N=60K, image features → scalar):

| Path | Wall-clock / generation | Wall-clock / 100-gen run |
|---|---|---|
| numpy | ~1 s | ~100 s |
| per-tree jit | ~50 ms (after warmup) | ~10 s |
| **batched vmap** | **~5-15 ms (after warmup)** | **~1-2 s** |

So a full MNIST GP run with Tier 1+2+3 should be ~seconds-to-minutes, not hours. That's tractable.

**Remaining bottlenecks for MNIST 95%:**

1. **Constant optimization (Nelder-Mead)** — still scipy + serial. Replace with `jax.grad + optax.adam` for another 5-50× win, and to enable vmap over the const-opt loop.
2. **MNIST feature evaluation** — 2-D `Measure2D` operators need a JAX path. Currently routes through `FunctionalOp2D.apply` which still uses numpy.
3. **Multi-feature ensemble (step 5)** — single-feature SR may plateau at ~90% on MNIST. The 95% target likely needs K discovered features + a linear layer trained via SGD.

**Next notebook:** `tessera_mnist.ipynb` — apply the full Tier 1+2+3 stack to MNIST and measure the actual accuracy ceiling. We'll discover whether 95% is reachable with single-feature SR or requires step 5.